In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS asb.olist_silver
;
CREATE SCHEMA IF NOT EXISTS asb.quarantine
;

In [0]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
run_id = str(uuid.uuid4())[:8]

def log_pipeline_run(entity_name, df_bronze, df_valid, df_quar):
    evaluated_count = df_bronze.count()
    valid_count = df_valid.count()
    quar_count = df_quar.count()
    
    audit_row = [(
        run_id,
        entity_name,
        evaluated_count,
        valid_count,
        quar_count
    )]
    
    schema = "run_id string, entity_name string, total_evaluated long, records_passed long, records_quarantined long"
    df_audit = spark.createDataFrame(audit_row, schema).withColumn("run_timestamp", F.current_timestamp())
    
    df_audit.write.format("delta").mode("append").saveAsTable("asb.quarantine.audit_pipeline_runs")

In [0]:
# ---------------------------------------------------------
# VALIDATE ORDERS
# ---------------------------------------------------------

df_orders = spark.table("asb.olist_bronze.raw_olist_orders_dataset")
window_order = Window.partitionBy("order_id")

df_orders_val = (
    df_orders
    .withColumn("_pk_count", F.count("order_id").over(window_order))
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(F.col("_pk_count") > 1, F.lit("ORD_UNIQ_01: Duplicate order_id")).otherwise(None),
        F.when(F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == ""), F.lit("ORD_COMP_01: Missing customer_id")).otherwise(None),
        F.when(~F.col("order_status").isin(['delivered', 'shipped', 'canceled', 'invoiced', 'processing', 'created', 'approved', 'unavailable']), 
               F.lit("ORD_VAL_01: Invalid order_status")).otherwise(None),
        F.when((F.col("order_delivered_customer_date").isNotNull()) & 
               (F.to_timestamp("order_delivered_customer_date") < F.to_timestamp("order_purchase_timestamp")), 
               F.lit("ORD_CONS_01: Delivery prior to purchase")).otherwise(None),
        F.when((F.col("order_approved_at").isNotNull()) & 
               (F.to_timestamp("order_approved_at") < F.to_timestamp("order_purchase_timestamp")), 
               F.lit("ORD_CONS_02: Approval prior to purchase")).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)

# Valid goes to silver layer
(
    df_orders_val.filter(F.col("_failure_count") == 0)
    .drop("_pk_count", "_dq_failures", "_failure_count")
    .write.format("delta").mode("overwrite").saveAsTable("asb.olist_silver.valid_orders")
)

# Quarantine invalid records
(
    df_orders_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("orders").alias("entity_name"),
        F.col("order_id").alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_orders")
)

# Log metrics
log_pipeline_run("orders", df_orders, df_orders_val.filter(F.col("_failure_count") == 0), df_orders_val.filter(F.col("_failure_count") > 0))

In [0]:
# ---------------------------------------------------------
# VALIDATE ORDER ITEMS
# ---------------------------------------------------------

df_items = spark.table("asb.olist_bronze.raw_olist_order_items_dataset")
window_item = Window.partitionBy("order_id", "order_item_id")

df_items_val = (
    df_items \
    .withColumn("_pk_count", F.count("*").over(window_item))
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(F.col("_pk_count") > 1, F.lit("ITM_UNIQ_01: Duplicate item line")).otherwise(None),
        F.when(F.col("price").cast("double") <= 0.0, F.lit("ITM_REAS_01: Non-positive price")).otherwise(None),
        F.when(F.col("freight_value").cast("double") < 0.0, F.lit("ITM_REAS_02: Negative freight")).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)

# Valid goes to silver layer
(
    df_items_val.filter(F.col("_failure_count") == 0)
    .drop("_pk_count", "_dq_failures", "_failure_count")
    .write.format("delta").mode("overwrite").saveAsTable("asb.olist_silver.valid_order_items")
)

# Quarantine invalid records
(
    df_items_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("order_items").alias("entity_name"),
        F.concat_ws("-", F.col("order_id"), F.col("order_item_id")).alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_order_items")
)

# Log metrics
log_pipeline_run("order_items", df_items, df_items_val.filter(F.col("_failure_count") == 0), df_items_val.filter(F.col("_failure_count") > 0))

In [0]:
# ---------------------------------------------------------
# VALIDATE ORDER PAYMENTS
# ---------------------------------------------------------

df_payments = spark.table("asb.olist_bronze.raw_olist_order_payments_dataset")

df_payments_val = (
    df_payments
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(~F.col("payment_type").isin(['credit_card', 'boleto', 'voucher', 'debit_card']), 
               F.lit("PAY_VAL_01: Invalid payment type")).otherwise(None),
        F.when(F.col("payment_value").cast("double") <= 0.0, 
               F.lit("PAY_REAS_01: Payment value <= 0")).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)

# Valid goes to silver layer
(
    df_payments_val.filter(F.col("_failure_count") == 0)
    .drop("_dq_failures", "_failure_count")
    .write.format("delta").mode("overwrite").saveAsTable("asb.olist_silver.valid_order_payments")
)

# Quarantine invalid records
(
    df_payments_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("order_payments").alias("entity_name"),
        F.concat_ws("-", F.col("order_id"), F.col("payment_sequential")).alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_order_payments")
)

# Log metrics
log_pipeline_run("order_payments", df_payments, df_payments_val.filter(F.col("_failure_count") == 0), df_payments_val.filter(F.col("_failure_count") > 0))

In [0]:
# ---------------------------------------------------------
# VALIDATE ORDER REVIEWS
# ---------------------------------------------------------

df_reviews = spark.table("asb.olist_bronze.raw_olist_order_reviews_dataset")
window_review = Window.partitionBy("review_id")

df_reviews_val = (
    df_reviews
    .withColumn("_pk_count", F.count("review_id").over(window_review))
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(F.col("_pk_count") > 1, F.lit("REV_UNIQ_01: Duplicate review_id")).otherwise(None),
        F.when(
            F.expr("try_cast(review_score AS INT) IS NULL OR try_cast(review_score AS INT) NOT BETWEEN 1 AND 5"),
            F.lit("REV_VAL_01: Score not between 1 and 5, or missing/malformed score")
        ).otherwise(None),
        F.when(
            F.expr("review_creation_date IS NOT NULL AND try_to_timestamp(review_creation_date) IS NULL"),
            F.lit("REV_VAL_02: Malformed review_creation_date")
        ).otherwise(None),
        F.when(
            F.expr("review_answer_timestamp IS NOT NULL AND try_to_timestamp(review_answer_timestamp) IS NULL"),
            F.lit("REV_VAL_03: Malformed review_answer_timestamp")
        ).otherwise(None),
        F.when(
            F.expr("try_to_timestamp(review_answer_timestamp) < try_to_timestamp(review_creation_date)"),
            F.lit("REV_CONS_01: Answer before creation")
        ).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)

# Valid goes to silver layer
(
    df_reviews_val.filter(F.col("_failure_count") == 0)
    .drop("_dq_failures", "_failure_count", "_pk_count")
    .write.format("delta").mode("overwrite").saveAsTable("asb.olist_silver.valid_order_reviews")
)

# Quarantine invalid records
(
    df_reviews_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("order_reviews").alias("entity_name"),
        F.col("review_id").alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_order_reviews")
)

# Log metrics
log_pipeline_run("order_reviews", df_reviews, df_reviews_val.filter(F.col("_failure_count") == 0), df_reviews_val.filter(F.col("_failure_count") > 0))

In [0]:
# ---------------------------------------------------------
# VALIDATE PRODUCTS
# ---------------------------------------------------------

df_products = spark.table("asb.olist_bronze.raw_olist_products_dataset")

df_products_val = (
    df_products
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(
            (F.col("product_weight_g").isNotNull()) & (F.col("product_weight_g").cast("double") <= 0.0), 
            F.lit("PROD_REAS_01: Non-positive weight")
        ).otherwise(None),
        F.when(
            (F.col("product_length_cm").isNotNull()) & (F.col("product_length_cm").cast("double") <= 0.0), 
            F.lit("PROD_REAS_01: Non-positive length")
        ).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)

# Valid goes to silver layer
(
    df_products_val.filter(F.col("_failure_count") == 0)
    .drop("_dq_failures", "_failure_count")
    .write.format("delta").mode("overwrite").saveAsTable("asb.olist_silver.valid_products")
)

# Quarantine invalid records
(
    df_products_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("products").alias("entity_name"),
        F.col("product_id").alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_products")
)

# Log metrics
log_pipeline_run("products", df_products, df_products_val.filter(F.col("_failure_count") == 0), df_products_val.filter(F.col("_failure_count") > 0))

In [0]:
# ---------------------------------------------------------
# VALIDATE CUSTOMERS & SELLERS
# ---------------------------------------------------------

# CUSTOMERS
df_customers = spark.table("asb.olist_bronze.raw_olist_customers_dataset")
df_cust_val = (
    df_customers
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(F.col("customer_unique_id").isNull() | (F.trim(F.col("customer_unique_id")) == ""), 
               F.lit("CUST_COMP_01: Null customer_unique_id")).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)
# Valid goes to silver layer
(
    df_cust_val
    .filter(F.col("_failure_count") == 0)
    .drop("_dq_failures", "_failure_count")
    .write.format("delta").mode("overwrite")
    .saveAsTable("asb.olist_silver.valid_customers")
)
# Quarantine invalid records
(
    df_cust_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("customers").alias("entity_name"),
        F.col("customer_id").alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_customers")
)
# Log metrics
log_pipeline_run("customers", df_customers, df_cust_val.filter(F.col("_failure_count") == 0), df_cust_val.filter(F.col("_failure_count") > 0))


# SELLERS
df_sellers = spark.table("asb.olist_bronze.raw_olist_sellers_dataset")
df_sellers_val = (
    df_sellers
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(F.col("seller_zip_code_prefix").isNull(), 
               F.lit("SELL_COMP_01: Null seller_zip_code_prefix")).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)
# Valid goes to silver layer
(
    df_sellers_val
    .filter(F.col("_failure_count") == 0)
    .drop("_dq_failures", "_failure_count")
    .write.format("delta").mode("overwrite")
    .saveAsTable("asb.olist_silver.valid_sellers")
)
# Quarantine invalid records
(
    df_sellers_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("sellers").alias("entity_name"),
        F.col("seller_id").alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_sellers")
)
# Log metrics
log_pipeline_run("sellers", df_sellers, df_sellers_val.filter(F.col("_failure_count") == 0), df_sellers_val.filter(F.col("_failure_count") > 0))

In [0]:
# ---------------------------------------------------------
# VALIDATE GEOLOCATION & LOAD TRANSLATIONS
# ---------------------------------------------------------
df_geo = spark.table("asb.olist_bronze.raw_olist_geolocation_dataset")

df_geo_val = (
    df_geo
    .withColumn("_dq_failures", F.array_compact(F.array(
        F.when(
            ~F.col("geolocation_lat").cast("double").between(-34.0, 5.5) | 
            ~F.col("geolocation_lng").cast("double").between(-74.0, -34.0),
            F.lit("GEO_VAL_01: Coordinates outside Brazilian geographic bounds")
        ).otherwise(None),
        F.when(F.col("geolocation_zip_code_prefix").isNull(),
               F.lit("GEO_COMP_01: Null zip code prefix")).otherwise(None)
    )))
    .withColumn("_failure_count", F.size(F.col("_dq_failures")))
)

# Valid goes to silver layer
(
    df_geo_val.filter(F.col("_failure_count") == 0)
    .drop("_dq_failures", "_failure_count")
    .write.format("delta").mode("overwrite").saveAsTable("asb.olist_silver.valid_geolocation")
)

# Quarantine invalid records
(
    df_geo_val.filter(F.col("_failure_count") > 0)
    .select(
        F.lit("geolocation").alias("entity_name"),
        F.col("geolocation_zip_code_prefix").alias("record_key"),
        F.col("_failure_count"),
        F.col("_dq_failures").alias("failed_rules"),
        F.to_json(F.struct("*")).alias("raw_payload"),
        F.current_timestamp().alias("_quarantined_at")
    ).write.format("delta").mode("overwrite").saveAsTable("asb.quarantine.quarantine_geolocation")
)

# Log metrics
log_pipeline_run("geolocation", df_geo, df_geo_val.filter(F.col("_failure_count") == 0), df_geo_val.filter(F.col("_failure_count") > 0))

In [0]:
# Lookup translations (Direct load into silver layer)
(
    spark.table("asb.olist_bronze.raw_product_category_name_translation")
    .write.format("delta").mode("overwrite").saveAsTable("asb.olist_silver.valid_product_category_translation")
)